# Reshaping and Pivoting 

## 1] *Reshaping with Hierarchical Indexing*
There are two primary actions :
- *stack* : This “rotates” or pivots from the columns in the data to the rows.

- *unstack* : This pivots from the rows into the columns

`Stack makes data taller. Unstack makes data wider.`

In [4]:
import numpy as np
import pandas as pd

data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(['Ohio', 'Colorado'], name='state'),
                    columns=pd.Index(['one', 'two', 'three'],
                    name='number'))

data


number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [5]:
result=data.stack()
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int64

In [6]:
result.unstack()

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [7]:
result.unstack(0)
# here 0 is level number it is realted to hierarchical index level

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


In [8]:
result.unstack('state')

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


In [9]:
s1 = pd.Series([0, 1, 2, 3], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([4, 5, 6], index=['c', 'd', 'e'])

data2=pd.concat([s1,s2],keys=['one','two'])
data2

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: int64

In [10]:
data2.unstack()

,a,b,c,d,e
one,0.0,1.0,2.0,3.0,NaN
two,NaN,NaN,4.0,5.0,6.0


In [11]:
data2.unstack().stack()

one  a    0.0
     b    1.0
     c    2.0
     d    3.0
     e    NaN
two  a    NaN
     b    NaN
     c    4.0
     d    5.0
     e    6.0
dtype: float64

In [12]:
data2.unstack().stack().dropna()

one  a    0.0
     b    1.0
     c    2.0
     d    3.0
two  c    4.0
     d    5.0
     e    6.0
dtype: float64

In [13]:
df = pd.DataFrame({'left': result, 'right': result + 5},
                columns=pd.Index(['left', 'right'], name='side'))

df


side             left  right
state    number             
Ohio     one        0      5
         two        1      6
         three      2      7
Colorado one        3      8
         two        4      9
         three      5     10

In [14]:
df.unstack('state')

side   left          right         
state  Ohio Colorado  Ohio Colorado
number                             
one       0        3     5        8
two       1        4     6        9
three     2        5     7       10

In [15]:
df.unstack('state').stack('side')

state         Ohio  Colorado
number side                 
one    left      0         3
       right     5         8
two    left      1         4
       right     6         9
three  left      2         5
       right     7        10

## Pivoting “Long” to “Wide” Format

In [16]:
ldata = pd.DataFrame({
    'date': ['2026-01','2026-01','2026-01',
             '2026-02','2026-02','2026-02'],
    'item': ['GDP','Inflation','Unemp',
             'GDP','Inflation','Unemp'],
    'value': [100,5,7,105,6,8]
})
ldata

,date,item,value
0,2026-01,GDP,100
1,2026-01,Inflation,5
2,2026-01,Unemp,7
3,2026-02,GDP,105
4,2026-02,Inflation,6
5,2026-02,Unemp,8


In [17]:
import statsmodels.api as sm
# Load the same macrodata dataset used in the book
# U.S. economic data
data = sm.datasets.macrodata.load_pandas().data

data.head()

,year,quarter,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint
0,1959.0,1.0,2710.349,1707.4,286.898,470.045,1886.9,28.98,139.7,2.82,5.8,177.146,0.00,0.00
1,1959.0,2.0,2778.801,1733.7,310.859,481.301,1919.7,29.15,141.7,3.08,5.1,177.830,2.34,0.74
2,1959.0,3.0,2775.488,1751.8,289.226,491.260,1916.4,29.35,140.5,3.82,5.3,178.657,2.74,1.09
3,1959.0,4.0,2785.204,1753.7,299.356,484.052,1931.3,29.37,140.0,4.33,5.6,179.386,0.27,4.06
4,1960.0,1.0,2847.699,1770.5,331.722,462.199,1955.5,29.54,139.6,3.50,5.2,180.007,2.31,1.19


In [18]:

periods = pd.PeriodIndex.from_fields(
    year=data['year'].astype(int),
    quarter=data['quarter'].astype(int),
    freq='Q'    #  Quarterly frequency (of upper 2 var from data)
)

data.index = periods.to_timestamp(how='end')

columns = pd.Index(['realgdp', 'infl', 'unemp'], name='item')
data = data.reindex(columns=columns)


# Remove time from date column
data.index = periods.to_timestamp('D', 'end').normalize()
ldata = data.stack().reset_index()

#Long Format
ldata.columns = ['date', 'item', 'value']
ldata[:10]

,date,item,value
0,1959-03-31,realgdp,2710.349
1,1959-03-31,infl,0.000
2,1959-03-31,unemp,5.800
3,1959-06-30,realgdp,2778.801
4,1959-06-30,infl,2.340
5,1959-06-30,unemp,5.100
6,1959-09-30,realgdp,2775.488
7,1959-09-30,infl,2.740
8,1959-09-30,unemp,5.300
9,1959-12-31,realgdp,2785.204


- `This code converts quarterly macroeconomic data from a wide time-series DataFrame`<br> 
`into a tidy long-format DataFrame by creating a quarterly PeriodIndex, converting it to date timestamps,`<br>
`selecting variables, stacking columns into rows, and resetting the index to obtain date, item, and value columns.`
## *Original* --> *Set date index* --> *Stack* --> *Reset index*

In [19]:

# Wide Format
pivoted = ldata.pivot(index='date', columns='item', values='value')
# Send  arguments always with keyword for pivot()
pivoted.head()

item,infl,realgdp,unemp
date,,,
1959-03-31,0.00,2710.349,5.8
1959-06-30,2.34,2778.801,5.1
1959-09-30,2.74,2775.488,5.3
1959-12-31,0.27,2785.204,5.6
1960-03-31,2.31,2847.699,5.2


In [20]:
ldata['value2'] = np.random.randn(len(ldata))
ldata[:10]

,date,item,value,value2
0,1959-03-31,realgdp,2710.349,0.399985
1,1959-03-31,infl,0.000,0.375490
2,1959-03-31,unemp,5.800,1.488672
3,1959-06-30,realgdp,2778.801,-0.238131
4,1959-06-30,infl,2.340,-0.169266
5,1959-06-30,unemp,5.100,-0.470696
6,1959-09-30,realgdp,2775.488,-0.289832
7,1959-09-30,infl,2.740,0.312781
8,1959-09-30,unemp,5.300,0.445040
9,1959-12-31,realgdp,2785.204,-0.811692


In [21]:
pivoted = ldata.pivot(index='date',columns= 'item')
pivoted[:5]

# here we got hierarchical index coz item has different separate values corresponding value and value2 

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-03-31  0.00  2710.349   5.8  0.375490  0.399985  1.488672
1959-06-30  2.34  2778.801   5.1 -0.169266 -0.238131 -0.470696
1959-09-30  2.74  2775.488   5.3  0.312781 -0.289832  0.445040
1959-12-31  0.27  2785.204   5.6 -1.516175 -0.811692  0.729328
1960-03-31  2.31  2847.699   5.2  1.181117 -1.411202 -0.762912

In [22]:
pivoted['value'][:5]

item,infl,realgdp,unemp
date,,,
1959-03-31,0.00,2710.349,5.8
1959-06-30,2.34,2778.801,5.1
1959-09-30,2.74,2775.488,5.3
1959-12-31,0.27,2785.204,5.6
1960-03-31,2.31,2847.699,5.2


##  `Pivot = set_index + unstack`
## pivot(index=..., columns=..., values=...) ≡ df.set_index([...]).unstack(...)
## both are equal

In [23]:
unstacked=ldata.set_index(['date','item']).unstack('item')
unstacked[:5]

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-03-31  0.00  2710.349   5.8  0.375490  0.399985  1.488672
1959-06-30  2.34  2778.801   5.1 -0.169266 -0.238131 -0.470696
1959-09-30  2.74  2775.488   5.3  0.312781 -0.289832  0.445040
1959-12-31  0.27  2785.204   5.6 -1.516175 -0.811692  0.729328
1960-03-31  2.31  2847.699   5.2  1.181117 -1.411202 -0.762912

# Example 2 : for pivoting `long` --> `wide` format

In [24]:

data = {
    "Student": ["Cummins", "Cummins", "Starc", "Starc", "Hazlewood", "Hazlewood"],
    "Subject": ["Math", "Science", "Math", "Science", "Math", "Science"],
    "Marks": [90, 85, 95, 88, 70, 75]
}

df = pd.DataFrame(data)
df


,Student,Subject,Marks
0,Cummins,Math,90
1,Cummins,Science,85
2,Starc,Math,95
3,Starc,Science,88
4,Hazlewood,Math,70
5,Hazlewood,Science,75


In [25]:
pivoted=df.pivot(index='Student',columns='Subject',values='Marks')
pivoted

Subject,Math,Science
Student,,
Cummins,90,85
Hazlewood,70,75
Starc,95,88


## Pivoting “Wide” to “Long” Format

In [26]:
df = pd.DataFrame({
    'User': ['Leo', 'Jeck', 'Oggy'],
    'Instagram': [120, 80, 95],
    'YouTube': [150, 200, 110],
    'WhatsApp': [60, 45, 70]
})
df

,User,Instagram,YouTube,WhatsApp
0,Leo,120,150,60
1,Jeck,80,200,45
2,Oggy,95,110,70


In [27]:
melted=pd.melt(df,['User'])
melted

,User,variable,value
0,Leo,Instagram,120
1,Jeck,Instagram,80
2,Oggy,Instagram,95
3,Leo,YouTube,150
4,Jeck,YouTube,200
5,Oggy,YouTube,110
6,Leo,WhatsApp,60
7,Jeck,WhatsApp,45
8,Oggy,WhatsApp,70


In [29]:
# Here we can give manual column names :

melted2 = pd.melt(
    df,
    id_vars='User',
    var_name='App',
    value_name='Minutes'
)

melted2

,User,App,Minutes
0,Leo,Instagram,120
1,Jeck,Instagram,80
2,Oggy,Instagram,95
3,Leo,YouTube,150
4,Jeck,YouTube,200
5,Oggy,YouTube,110
6,Leo,WhatsApp,60
7,Jeck,WhatsApp,45
8,Oggy,WhatsApp,70


In [34]:
#  Reverse process 
reshaped=melted2.pivot(index='User',columns='App',values='Minutes')
reshaped

App,Instagram,WhatsApp,YouTube
User,,,
Jeck,80,45,200
Leo,120,60,150
Oggy,95,70,110


In [35]:
reshaped.reset_index()

App,User,Instagram,WhatsApp,YouTube
0,Jeck,80,45,200
1,Leo,120,60,150
2,Oggy,95,70,110


In [36]:
pd.melt(df,id_vars=['User'],value_vars=['YouTube','WhatsApp'])

,User,variable,value
0,Leo,YouTube,150
1,Jeck,YouTube,200
2,Oggy,YouTube,110
3,Leo,WhatsApp,60
4,Jeck,WhatsApp,45
5,Oggy,WhatsApp,70


In [ ]:
# without grp identifiers 
pd.melt(df,value_vars=['Instagram','YouTube','WhatsApp'])

,variable,value
0,Instagram,120
1,Instagram,80
2,Instagram,95
3,YouTube,150
4,YouTube,200
5,YouTube,110
6,WhatsApp,60
7,WhatsApp,45
8,WhatsApp,70


In [39]:
pd.melt(df,value_vars=['User','Instagram','YouTube'])

,variable,value
0,User,Leo
1,User,Jeck
2,User,Oggy
3,Instagram,120
4,Instagram,80
5,Instagram,95
6,YouTube,150
7,YouTube,200
8,YouTube,110
